# Speech Coach — GPU Backend Server

**Run this notebook on Colab with a T4 GPU runtime.**

It starts a FastAPI server that your local webapp calls for pipeline processing.

### Setup (one-time):
1. Open this notebook in Google Colab
2. Runtime → Change runtime type → **T4 GPU**
3. **Run All** (Ctrl+F9)
4. Copy the ngrok URL printed at the bottom
5. Paste it in your webapp Settings → COLAB_BACKEND_URL

After that, all uploads from `localhost:3000` are processed on the T4 GPU automatically.

In [ ]:
#@title 1. Install Dependencies
# Colab already has PyTorch+CUDA, so don't reinstall it
!pip install -q transformers librosa noisereduce
!pip install -q praat-parselmouth
!pip install -q openai-whisper
!pip install -q opencv-python mediapipe ultralytics
!pip install -q spacy textstat sentence-transformers
!pip install -q speechbrain
!pip install -q language_tool_python
!pip install -q fastapi uvicorn python-multipart pyngrok
!python -m spacy download en_core_web_sm -q
print('✅ Dependencies installed')

In [ ]:
#@title 2. Clone / Update Repo
import os

REPO_URL = 'https://github.com/anvay-cpu/voice-analysis-pipeline.git'
REPO_DIR = '/content/voice-analysis-pipeline'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'✅ Repo ready at {REPO_DIR}')

In [ ]:
#@title 3. Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print('✅ T4 GPU ready')
else:
    print('⚠️ No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
#@title 4. Setup ngrok tunnel
#@markdown Get a free auth token at https://dashboard.ngrok.com/signup
NGROK_AUTH_TOKEN = '' #@param {type:"string"}

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print('✅ ngrok authenticated')
else:
    print('⚠️ No ngrok token — tunnel will work but may be rate-limited')
    print('   Get a free token at: https://dashboard.ngrok.com/signup')

In [ ]:
#@title 4b. Connect Claude Proxy (for LLM features)
#@markdown Paste the ngrok URL from your local Claude proxy here.
#@markdown Run `python scripts/claude_proxy.py --ngrok` on your local machine first.
CLAUDE_PROXY_URL = '' #@param {type:"string"}

import os
if CLAUDE_PROXY_URL:
    os.environ['CLAUDE_PROXY_URL'] = CLAUDE_PROXY_URL.rstrip('/')
    # Verify connection
    import requests
    try:
        r = requests.get(f"{CLAUDE_PROXY_URL.rstrip('/')}/health", timeout=5)
        if r.ok:
            print(f'✅ Claude proxy connected: {CLAUDE_PROXY_URL}')
            print('   LLM features (tone, regime, argument, coaching) will use Claude via your Max subscription')
        else:
            print(f'⚠️ Proxy responded with status {r.status_code}')
    except Exception as e:
        print(f'⚠️ Could not reach proxy: {e}')
        print('   LLM features will use heuristic fallbacks (still works, lower quality)')
else:
    print('ℹ️ No Claude proxy URL — LLM features will use heuristic fallbacks')
    print('   To enable: run `python scripts/claude_proxy.py --ngrok` locally and paste the URL above')

In [ ]:
#@title 5. Pre-load Models (warm up GPU)
import sys
sys.path.insert(0, REPO_DIR)

print('Loading master pipeline...')
from src.master_pipeline import SpeechCoachPipeline
master = SpeechCoachPipeline()

print('Loading voice pipeline...')
try:
    from src.pipeline import VoiceAnalysisPipeline
    master._voice_pipeline = VoiceAnalysisPipeline()
    print('  ✅ Voice pipeline ready')
except Exception as e:
    print(f'  ⚠️ Voice pipeline: {e}')

print('Loading body pipeline...')
try:
    from src.body.pipeline import BodyAnalysisPipeline
    master._body_pipeline = BodyAnalysisPipeline()
    print('  ✅ Body pipeline ready')
except Exception as e:
    print(f'  ⚠️ Body pipeline: {e}')

print('Loading content pipeline...')
try:
    from src.content.pipeline import ContentAnalysisPipeline
    master._content_pipeline = ContentAnalysisPipeline()
    print('  ✅ Content pipeline ready')
except Exception as e:
    print(f'  ⚠️ Content pipeline: {e}')

# Inject the pre-loaded pipeline into the API server so it reuses these models
from src.api_server import set_pipeline
set_pipeline(master)

print('\n✅ All models loaded on GPU and injected into API server')

In [ ]:
#@title 5a. Test: Create a 10-second test video
#@markdown Creates a tiny synthetic test video so we can validate the full pipeline.
import numpy as np
import cv2, subprocess, os

TEST_VIDEO = '/content/test_10s.mp4'

# Create a 10-second 320x240 video with a moving rectangle (simulates a speaker)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter('/content/_temp.mp4', fourcc, 15, (320, 240))
for i in range(150):  # 10s at 15fps
    frame = np.zeros((240, 320, 3), dtype=np.uint8)
    x = 80 + int(40 * np.sin(i * 0.1))
    cv2.rectangle(frame, (x, 40), (x+160, 200), (200, 180, 160), -1)
    cv2.circle(frame, (x+80, 80), 25, (220, 200, 180), -1)  # head
    writer.write(frame)
writer.release()

# Add a sine-wave audio track (simulates speech with varying frequency)
sr = 16000
t = np.linspace(0, 10, sr * 10)
# Mix frequencies to simulate speech-like audio
audio = 0.3 * np.sin(2 * np.pi * 200 * t) + 0.2 * np.sin(2 * np.pi * 400 * t * (1 + 0.3*np.sin(0.5*t)))
audio = (audio * 32767).astype(np.int16)
import wave
with wave.open('/content/_temp.wav', 'w') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sr)
    wf.writeframes(audio.tobytes())

# Mux video + audio
subprocess.run(['ffmpeg', '-y', '-i', '/content/_temp.mp4', '-i', '/content/_temp.wav',
                '-c:v', 'libx264', '-c:a', 'aac', '-shortest', TEST_VIDEO],
               capture_output=True)

# Cleanup temp
os.remove('/content/_temp.mp4')
os.remove('/content/_temp.wav')

print(f'✅ Test video: {TEST_VIDEO} ({os.path.getsize(TEST_VIDEO)/1024:.0f} KB)')

In [ ]:
#@title 5b. Test: Run each pipeline step on test video
#@markdown Validates voice → body → content → fusion → scoring → coaching → charts → report
import traceback
TEST_VIDEO = "/content/test_10s.mp4"
errors = []

# --- Step 1: Voice ---
print("=" * 50)
print("[1/8] VOICE PIPELINE")
try:
    voice_output = master._run_voice_pipeline(TEST_VIDEO)
    has_windows = len(voice_output.get("windows", [])) > 0
    has_transcript = bool(voice_output.get("transcript", {}).get("text", ""))
    duration = voice_output.get("metadata", {}).get("duration_sec", 0)
    print(f"  ✅ windows={len(voice_output.get('windows', []))}, transcript={has_transcript}, duration={duration:.1f}s")
except Exception as e:
    errors.append(("VOICE", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    voice_output = {"windows": [], "summary": {}, "metadata": {"duration_sec": 10}, "transcript": {"text": "", "words": [], "segments": []}}

# --- Step 2: Body ---
print("\n[2/8] BODY PIPELINE")
try:
    body_output = master._run_body_pipeline(TEST_VIDEO)
    n_segs = len(body_output.get("segments", []))
    print(f"  ✅ segments={n_segs}, keys={list(body_output.keys())}")
except Exception as e:
    errors.append(("BODY", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    body_output = {"segments": [], "summary": {}}

# --- Step 3: Content ---
print("\n[3/8] CONTENT PIPELINE")
try:
    content_output = master._run_content_pipeline(voice_output)
    n_segs = len(content_output.get("segments", []))
    print(f"  ✅ segments={n_segs}, keys={list(content_output.keys())}")
except Exception as e:
    errors.append(("CONTENT", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    traceback.print_exc()
    content_output = {"segments": [], "summary": {}}

# --- Step 4: Fusion ---
print("\n[4/8] FUSION ENGINE")
try:
    duration = master._get_duration(voice_output, body_output)
    voice_windows = master._extract_voice_windows(voice_output)
    fusion_output = master.fusion_engine.fuse(voice_windows, body_output, content_output, duration)
    tl = len(fusion_output.get("timeline", []))
    print(f"  ✅ timeline={tl}pts, boundaries={len(fusion_output.get('regime_boundaries', []))}")
except Exception as e:
    errors.append(("FUSION", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    traceback.print_exc()
    fusion_output = {"timeline": [], "duration_sec": 10, "regime_boundaries": [], "transitions": {}, "recovery": {"total_disruptions": 0, "mean_composure": 1.0, "disruptions": []}, "emotion_coherence": {"coherence_score_0_100": 50, "mean_coherence": 0.5}}

# --- Step 5: Scoring ---
print("\n[5/8] DIMENSION SCORING")
try:
    scores = master.scorer.score_all(fusion_output)
    print(f"  ✅ overall={scores.get('overall', 0):.0f}/100")
    for k, v in scores.items():
        if k != "overall": print(f"      {k}: {v:.0f}")
except Exception as e:
    errors.append(("SCORING", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    traceback.print_exc()
    scores = {d: 50.0 for d in ["vocal_clarity","body_language","content_structure","audience_engagement","emotional_expressiveness","regime_adaptability","overall"]}

# --- Step 6: Coaching ---
print("\n[6/8] COACHING GENERATION")
try:
    coaching = master.coaching_writer.generate_full_report(
        all_segments=[], overall_scores=scores,
        fusion_output=fusion_output,
        speech_metadata={"duration_sec": duration, "video_path": TEST_VIDEO})
    print(f"  ✅ summary={len(coaching.get('executive_summary', ''))} chars, feedback={len(coaching.get('segment_feedback', []))}, exercises={len(coaching.get('practice_plan', []))}")
except Exception as e:
    errors.append(("COACHING", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    traceback.print_exc()
    coaching = {"executive_summary": "Analysis complete.", "segment_feedback": [], "practice_plan": []}

# --- Step 7: Charts ---
print("\n[7/8] CHART GENERATION")
try:
    import os; os.makedirs("/content/test_charts", exist_ok=True)
    master.chart_generator.output_dir = "/content/test_charts"
    charts = master.chart_generator.generate_all(scores, fusion_output["timeline"], fusion_output)
    ok = sum(1 for v in charts.values() if v is not None)
    print(f"  ✅ {ok}/{len(charts)} charts generated")
    for name, path in charts.items():
        status = "✅" if path else "⚠️ skipped"
        print(f"      {name}: {status}")
except Exception as e:
    errors.append(("CHARTS", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    traceback.print_exc()
    charts = {}

# --- Step 8: Report Transform ---
print("\n[8/8] REPORT TRANSFORMER")
try:
    from src.report_transformer import transform_pipeline_output
    report = transform_pipeline_output(
        voice_output=voice_output, body_output=body_output,
        content_output=content_output, fusion_output=fusion_output,
        scores=scores, coaching=coaching,
        session_id="TEST_001", title="Test Speech",
        processing_time_sec=0.0)
    required_keys = ["id","title","scores","timeline","coaching_feedback","practice_plan","executive_summary","transcript"]
    missing = [k for k in required_keys if k not in report]
    print(f"  ✅ Report has {len(report)} keys, timeline={len(report.get('timeline', []))} pts")
    if missing: print(f"  ⚠️ Missing keys: {missing}")
except Exception as e:
    errors.append(("REPORT", str(e)))
    print(f"  ❌ {type(e).__name__}: {e}")
    traceback.print_exc()

# --- Summary ---
print("\n" + "=" * 50)
if errors:
    print(f"⚠️ {len(errors)} step(s) had errors:")
    for step, err in errors:
        print(f"  [{step}] {err}")
    print("\nThe pipeline will still work — failed steps use fallback data.")
else:
    print("✅ ALL 8 STEPS PASSED — Pipeline is ready!")


In [ ]:
#@title 5c. Test: Validate report JSON matches webapp types
#@markdown Ensures the generated report matches the TypeScript SpeechReport interface.
import json

EXPECTED_SCHEMA = {
    "id": str, "title": str, "duration_sec": (int, float),
    "processing_time_sec": (int, float),
    "scores": dict, "content_details": dict, "fusion_stats": dict,
    "executive_summary": str, "regime_segments": list,
    "regime_transitions": list, "disruption_events": list,
    "coaching_feedback": list, "practice_plan": list,
    "filler_words": list, "transcript": str,
    "timeline": list, "emotion_arc": list, "heatmap_channels": list,
}

SCORE_DIMS = ["vocal_clarity","body_language","content_structure",
              "audience_engagement","emotional_expressiveness",
              "regime_adaptability","overall"]

all_ok = True
try:
    for key, expected_type in EXPECTED_SCHEMA.items():
        if key not in report:
            print(f"  ❌ Missing key: {key}")
            all_ok = False
        elif not isinstance(report[key], expected_type):
            print(f"  ❌ {key}: expected {expected_type}, got {type(report[key])}")
            all_ok = False

    # Check scores sub-keys
    for dim in SCORE_DIMS:
        if dim not in report.get("scores", {}):
            print(f"  ❌ Missing score: {dim}")
            all_ok = False

    # Check content_details
    for key in ["grammar_score","fkgl","ttr","dominant_tone","dominant_sentiment"]:
        if key not in report.get("content_details", {}):
            print(f"  ❌ Missing content_details.{key}")
            all_ok = False

    # Validate a timeline point structure if exists
    if report.get("timeline"):
        tp = report["timeline"][0]
        for sub in ["voice", "body", "content"]:
            if sub not in tp:
                print(f"  ❌ timeline[0] missing .{sub}")
                all_ok = False

    # Print summary
    if all_ok:
        print("✅ Report JSON matches SpeechReport interface")
        print(f"  Scores: {json.dumps(report['scores'], indent=2)}")
        print(f"  Timeline points: {len(report['timeline'])}")
        print(f"  Coaching items: {len(report['coaching_feedback'])}")
        print(f"  Practice exercises: {len(report['practice_plan'])}")
        print(f"  Summary: {report['executive_summary'][:100]}...")
    else:
        print("\n⚠️ Report has schema issues — webapp may not render correctly")
except NameError:
    print("⚠️ No report object — run cell 5b first")


In [ ]:
#@title 6. Start GPU Backend Server 🚀
#@markdown This cell runs forever — the server stays up as long as Colab is connected.

import threading
import asyncio
import uvicorn
from pyngrok import ngrok

# Import the API server
from src.api_server import app

# Create ngrok tunnel
public_url = ngrok.connect(8000, "http")
public_url_str = str(public_url).split('"')[1] if '"' in str(public_url) else str(public_url)

print('=' * 60)
print('  🚀 SPEECH COACH GPU BACKEND IS LIVE')
print('=' * 60)
print(f'')
print(f'  📡 Public URL: {public_url_str}')
print(f'')
print(f'  Paste this URL in your webapp:')
print(f'  Settings → COLAB_BACKEND_URL → {public_url_str}')
print(f'')
print('=' * 60)

# Run uvicorn in its own thread with its own event loop
def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# Keep cell alive
import time
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('\n🛑 Server stopped')
